In [ ]:
import os
from PIL import Image

# input and output folders
input_folder = r"F:\strwaberry images\Front_row_with_text"
output_base = r"F:\strwaberry images\Front_row_with_text\Cropped_Images"

# cropping coordinates: (x1, y1, x2, y2)
crops = {
    "down_1": (15, 423, 722,928),
    "down_2": (909,268, 1503,884),
    "down_3": (1675,470, 2136, 926),
    "down_4": (2405, 418, 2961,911),
    "down_5": (3237,431, 3809, 867),
}

# create output folders
for folder in crops.keys():
    os.makedirs(os.path.join(output_base, folder), exist_ok=True)

# process images
for img_name in os.listdir(input_folder):
    if img_name.lower().endswith((".jpg", ".jpeg", ".png")):
        img_path = os.path.join(input_folder, img_name)
        image = Image.open(img_path)

        for folder_name, (x1, y1, x2, y2) in crops.items():
            cropped = image.crop((x1, y1, x2, y2))

            save_path = os.path.join(output_base, folder_name, img_name)
            cropped.save(save_path)

print("Cropping completed")


Cropping completed


In [5]:

##back row cropping
import os
from PIL import Image

# input and output folders
input_folder = r"F:\strwaberry images\back row upated"
output_base = r"F:\strwaberry images\back row upated with text\Cropped_Images"
crops = {
"down_2": (710,894,1593,1498),
"down_4": (3659,747, 4588,1571)
}

# create output folders
for folder in crops.keys():
    os.makedirs(os.path.join(output_base, folder), exist_ok=True)

# process images
for img_name in os.listdir(input_folder):
    if img_name.lower().endswith((".jpg", ".jpeg", ".png")):
        img_path = os.path.join(input_folder, img_name)
        image = Image.open(img_path)

        for folder_name, (x1, y1, x2, y2) in crops.items():
            cropped = image.crop((x1, y1, x2, y2))

            save_path = os.path.join(output_base, folder_name, img_name)
            cropped.save(save_path)

print("Cropping completed")

Cropping completed


In [8]:
import cv2
import os
from pathlib import Path

input_folder = "F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1"          # your folder
output_folder = "F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1\CLAHE"  # output folder
os.makedirs(output_folder, exist_ok=True)

def apply_clahe_illumination(image_path, save_path):
    img = cv2.imread(image_path)

    # Convert to LAB color space
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    # CLAHE only on Lightness channel
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8))
    cl = clahe.apply(l)

    # Merge back
    merged = cv2.merge((cl, a, b))
    final = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

    cv2.imwrite(save_path, final)


# Process all images
for img_path in Path(input_folder).glob("*"):
    save_path = os.path.join(output_folder, img_path.name)
    apply_clahe_illumination(str(img_path), save_path)

print("CLAHE preprocessing completed")

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'


In [9]:
import cv2
import numpy as np
import os
from pathlib import Path

# ===================== DEHAZE =====================
def dehaze(img):
    img = img.astype(np.float32) / 255.0

    dark = np.min(img, axis=2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    dark = cv2.erode(dark, kernel)

    A = np.max(img[dark >= np.percentile(dark, 99)])
    t = 1 - 0.95 * dark / A
    t = np.clip(t, 0.2, 1)

    J = np.zeros_like(img)
    for c in range(3):
        J[:, :, c] = (img[:, :, c] - A) / t + A

    return np.clip(J * 255, 0, 255).astype(np.uint8)

# ===================== CLAHE (LAB) =====================
def clahe_lab(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    l = clahe.apply(l)

    lab = cv2.merge((l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

# ===================== FULL PIPELINE =====================
def normalize_image(img):
    img = dehaze(img)      # remove sunlight veil
    img = clahe_lab(img)   # normalize illumination
    return img

# ===================== BATCH PROCESS =====================
input_folder = "F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1"          # your folder
output_folder = "F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1\CLAHE_Dehaze"  # output folder
os.makedirs(output_folder, exist_ok=True)

for img_path in Path(input_folder).glob("*"):
    img = cv2.imread(str(img_path))
    if img is None:
        continue

    processed = normalize_image(img)
    save_path = os.path.join(output_folder, img_path.name)
    cv2.imwrite(save_path, processed)

print("✅ All images normalized and saved")

✅ All images normalized and saved


In [2]:
import cv2
import numpy as np
import os
from pathlib import Path

# -------- COLOR MATCH FUNCTION --------
def color_transfer(source, target):
    source = cv2.cvtColor(source, cv2.COLOR_BGR2LAB).astype("float32")
    target = cv2.cvtColor(target, cv2.COLOR_BGR2LAB).astype("float32")

    (lMeanSrc, aMeanSrc, bMeanSrc) = cv2.mean(source)[:3]
    (lStdSrc, aStdSrc, bStdSrc) = source.std(axis=(0,1))

    (lMeanTar, aMeanTar, bMeanTar) = cv2.mean(target)[:3]
    (lStdTar, aStdTar, bStdTar) = target.std(axis=(0,1))

    l, a, b = cv2.split(source)

    l = ((l - lMeanSrc) * (lStdTar / (lStdSrc+1e-6))) + lMeanTar
    a = ((a - aMeanSrc) * (aStdTar / (aStdSrc+1e-6))) + aMeanTar
    b = ((b - bMeanSrc) * (bStdTar / (bStdSrc+1e-6))) + bMeanTar

    result = cv2.merge([l, a, b])
    result = np.clip(result, 0, 255).astype("uint8")
    return cv2.cvtColor(result, cv2.COLOR_LAB2BGR)

# -------- LOAD REFERENCE --------
reference = cv2.imread(r"F:\strwaberry images\back row upated with text\Cropped_Images\down_3\citra_2025-12-15_154501.jpg")

input_folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\down_3"         # your folder
output_folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\down_3\preprocessed_images"  # output folder
os.makedirs(output_folder, exist_ok=True)

# -------- PROCESS ALL IMAGES --------
for img_path in Path(input_folder).glob("*"):
    img = cv2.imread(str(img_path))
    if img is None:
        continue

    normalized = color_transfer(img, reference)
    cv2.imwrite(os.path.join(output_folder, img_path.name), normalized)

print("✔ Dataset normalized to reference lighting")

✔ Dataset normalized to reference lighting


In [7]:
import cv2
import numpy as np
import os
from pathlib import Path

# -------- COLOR MATCH FUNCTION --------
def color_transfer(source, target):
    source = cv2.cvtColor(source, cv2.COLOR_BGR2LAB).astype("float32")
    target = cv2.cvtColor(target, cv2.COLOR_BGR2LAB).astype("float32")

    (lMeanSrc, aMeanSrc, bMeanSrc) = cv2.mean(source)[:3]
    (lStdSrc, aStdSrc, bStdSrc) = source.std(axis=(0,1))

    (lMeanTar, aMeanTar, bMeanTar) = cv2.mean(target)[:3]
    (lStdTar, aStdTar, bStdTar) = target.std(axis=(0,1))

    l, a, b = cv2.split(source)

    l = ((l - lMeanSrc) * (lStdTar / (lStdSrc+1e-6))) + lMeanTar
    a = ((a - aMeanSrc) * (aStdTar / (aStdSrc+1e-6))) + aMeanTar
    b = ((b - bMeanSrc) * (bStdTar / (bStdSrc+1e-6))) + bMeanTar

    result = cv2.merge([l, a, b])
    result = np.clip(result, 0, 255).astype("uint8")
    return cv2.cvtColor(result, cv2.COLOR_LAB2BGR)

# -------- LOAD REFERENCE --------
reference = cv2.imread(r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2\citra_2025-12-16_134501.jpg")

input_folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2"          # your folder
output_folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images"  # output folder
os.makedirs(output_folder, exist_ok=True)

# -------- PROCESS ALL IMAGES --------
for img_path in Path(input_folder).glob("*"):
    img = cv2.imread(str(img_path))
    if img is None:
        continue

    normalized = color_transfer(img, reference)
    cv2.imwrite(os.path.join(output_folder, img_path.name), normalized)

print("✔ Dataset normalized to reference lighting")

✔ Dataset normalized to reference lighting


In [8]:
!pip install plotly

In [2]:
!pip install opencv-python plotly

  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)


In [3]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px
from datetime import datetime

folder = r"F:\strwaberry images\Back_Row_with_text\Cropped_Images\down_2\preprocessed_images"

records = []

for path in sorted(Path(folder).glob("*.jpg")):

    # ---------- READ IMAGE ----------
    img = cv2.imread(str(path))
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # ---------- GREEN MASK ----------
    lower_green = np.array([35, 40, 40])
    upper_green = np.array([90, 255, 255])

    mask = cv2.inRange(hsv, lower_green, upper_green)

    # remove noise
    kernel = np.ones((5,5),np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    green_pixels = cv2.countNonZero(mask)

    # ---------- EXTRACT DATE ----------
    # filename: citra_2026-01-03_12-10.jpg
    date_str = path.stem.split("_")[1]      # 2026-01-03
    date = datetime.strptime(date_str, "%Y-%m-%d").date()

    records.append([date, green_pixels])

# ---------- DATAFRAME ----------
df = pd.DataFrame(records, columns=["date", "green_pixels"])

# average per day (removes lighting noise)
daily_df = df.groupby("date").mean().reset_index()

# ---------- INTERACTIVE PLOT ----------
fig = px.line(
    daily_df,
    x="date",
    y="green_pixels",
    title="Daily Strawberry Plant Growth (Green Leaf Area)",
    markers=True
)

fig.show()

In [8]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px
from datetime import datetime

folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images"

records = []

for path in sorted(Path(folder).glob("*.jpg")):

    # ---------- READ IMAGE ----------
    img = cv2.imread(str(path))
    if img is None:
        continue

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # ---------- GREEN MASK ----------
    lower_green = np.array([35, 40, 40])
    upper_green = np.array([90, 255, 255])

    mask = cv2.inRange(hsv, lower_green, upper_green)

    kernel = np.ones((5,5),np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    green_pixels = cv2.countNonZero(mask)

    # ---------- EXTRACT FULL DATETIME ----------
    # citra_2025-12-19_144501
    name = path.stem.replace("citra_", "")
    dt = datetime.strptime(name, "%Y-%m-%d_%H%M%S")

    records.append([dt, green_pixels])

# ---------- DATAFRAME ----------
df = pd.DataFrame(records, columns=["datetime", "green_pixels"])

# STEP 1: average per minute (ignore seconds)
df["minute"] = df["datetime"].dt.floor("min")
minute_df = df.groupby("minute").mean().reset_index()

# STEP 2: average per day
minute_df["date"] = minute_df["minute"].dt.date
daily_df = minute_df.groupby("date").mean().reset_index()

# ---------- INTERACTIVE PLOT ----------
fig = px.line(
    daily_df,
    x="date",
    y="green_pixels",
    title="Daily Strawberry Plant Growth (Green Leaf Area)",
    markers=True
)

fig.show()



In [1]:
import cv2
import os

# paths
orig_dir = "F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1"
norm_dir = "F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1\preprocessed_images"
out_dir = "F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1_comparison"

os.makedirs(out_dir, exist_ok=True)

# loop through images
for name in os.listdir(orig_dir):

    orig_path = os.path.join(orig_dir, name)
    norm_path = os.path.join(norm_dir, name)

    # skip if normalized image not found
    if not os.path.exists(norm_path):
        print(f"Missing normalized for {name}")
        continue

    # read images
    img_orig = cv2.imread(orig_path)
    img_norm = cv2.imread(norm_path)

    if img_orig is None or img_norm is None:
        print(f"Error reading {name}")
        continue

    # resize to same height
    h = min(img_orig.shape[0], img_norm.shape[0])

    def resize_keep_ratio(img, height):
        ratio = height / img.shape[0]
        width = int(img.shape[1] * ratio)
        return cv2.resize(img, (width, height))

    img_orig = resize_keep_ratio(img_orig, h)
    img_norm = resize_keep_ratio(img_norm, h)

    # add labels
    cv2.putText(img_orig, "Original", (20,40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.putText(img_norm, "Normalized", (20,40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    # combine horizontally
    combined = cv2.hconcat([img_orig, img_norm])

    # save
    save_path = os.path.join(out_dir, name)
    cv2.imwrite(save_path, combined)

print("Done! Images saved in comparison folder")

Missing normalized for 2025-11-27_13-34.jpg
Missing normalized for 2025-11-27_13-35.jpg
Missing normalized for 2025-12-12_14-45.jpg
Missing normalized for CLAHE
Missing normalized for CLAHE_Dehaze
Error reading files.txt
Error reading front_row_donw_plant1.mp4
Missing normalized for preprocessed_images
Done! Images saved in comparison folder


In [2]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px
from datetime import datetime

folder = r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images"

records = []

for path in sorted(Path(folder).glob("*.jpg")):

    # ---------- READ IMAGE ----------
    img = cv2.imread(str(path))
    if img is None:
        continue

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # ---------- GREEN MASK ----------
    lower_green = np.array([35, 40, 40])
    upper_green = np.array([90, 255, 255])

    mask = cv2.inRange(hsv, lower_green, upper_green)

    kernel = np.ones((5,5),np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    green_pixels = cv2.countNonZero(mask)

    # ---------- EXTRACT FULL DATETIME ----------
    # citra_2025-12-19_144501
    name = path.stem.replace("citra_", "")
    dt = datetime.strptime(name, "%Y-%m-%d_%H%M%S")

    records.append([dt, green_pixels])

# ---------- DATAFRAME ----------
df = pd.DataFrame(records, columns=["datetime", "green_pixels"])

# Sort by time (important!)
df = df.sort_values("datetime")

# ---------- INTERACTIVE PLOT (EVERY IMAGE) ----------
fig = px.line(
    df,
    x="datetime",
    y="green_pixels",
    title="Strawberry Plant Growth Per Image (Green Leaf Area)",
    markers=True
)

fig.update_layout(
    xaxis_title="Time",
    yaxis_title="Green Pixels (Leaf Area)",
    hovermode="x unified"
)

fig.show()

In [1]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px

def compare_chlorophyll_histogram(original_folder, preprocessed_folder):
    """
    Compare chlorophyll (green pixel area) distribution
    between original and normalized (preprocessed) images.
    
    Parameters:
    -----------
    original_folder : str
        Path to original images
    preprocessed_folder : str
        Path to normalized/preprocessed images
    """

    def compute_green_pixels(folder_path):
        green_values = []

        for path in sorted(Path(folder_path).glob("*.jpg")):
            img = cv2.imread(str(path))
            if img is None:
                continue

            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

            lower_green = np.array([35, 40, 40])
            upper_green = np.array([90, 255, 255])

            mask = cv2.inRange(hsv, lower_green, upper_green)

            kernel = np.ones((5,5), np.uint8)
            mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
            mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

            green_pixels = cv2.countNonZero(mask)
            green_values.append(green_pixels)

        return green_values

    # Compute values
    original_green = compute_green_pixels(original_folder)
    preprocessed_green = compute_green_pixels(preprocessed_folder)

    # Create dataframe for plotting
    df = pd.DataFrame({
        "Green Pixels": original_green + preprocessed_green,
        "Image Type": ["Original"] * len(original_green) +
                      ["Preprocessed"] * len(preprocessed_green)
    })

    # Summary statistics
    print("\n--- Summary Statistics ---")
    print(df.groupby("Image Type")["Green Pixels"].describe())

    # Plot histogram comparison
    fig = px.histogram(
        df,
        x="Green Pixels",
        color="Image Type",
        barmode="overlay",
        opacity=0.6,
        nbins=40,
        title="Histogram Comparison: Original vs Preprocessed Chlorophyll Area"
    )

    fig.show()


compare_chlorophyll_histogram(
    r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2",
    r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images"
)


--- Summary Statistics ---
               count           mean           std      min        25%  \
Image Type                                                              
Original      1558.0  129310.136714  46882.081389      0.0   94221.75   
Preprocessed  1558.0  137592.017330  24834.899990  76075.0  117213.75   

                   50%        75%       max  
Image Type                                   
Original      130350.5  167383.00  246395.0  
Preprocessed  138512.5  157145.25  197227.0  


In [ ]:
# More concentrated distribution

# Most values lie between ~90k and ~170k

# Fewer extreme values

# Distribution appears smoother and more centered

In [2]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px

def compare_normalization_histogram(original_folder, preprocessed_folder):
    """
    Compare brightness distribution of original vs normalized images
    to check whether lighting variation has been reduced.
    """

    def compute_brightness(folder):
        brightness_values = []

        for path in sorted(Path(folder).glob("*.jpg")):
            img = cv2.imread(str(path))
            if img is None:
                continue

            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
            v_channel = hsv[:, :, 2]   # brightness channel

            brightness_values.extend(v_channel.flatten())

        return brightness_values

    # Extract brightness values
    original_brightness = compute_brightness(original_folder)
    preprocessed_brightness = compute_brightness(preprocessed_folder)

    df = pd.DataFrame({
        "Pixel Intensity": original_brightness + preprocessed_brightness,
        "Image Type": ["Original"] * len(original_brightness) +
                      ["Preprocessed"] * len(preprocessed_brightness)
    })

    # Histogram comparison
    fig = px.histogram(
        df,
        x="Pixel Intensity",
        color="Image Type",
        barmode="overlay",
        opacity=0.6,
        nbins=60,
        title="Brightness Distribution: Original vs Normalized Images"
    )

    fig.show()


compare_normalization_histogram(
    r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2",
    r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images"
)



MemoryError: 

In [3]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px

def compare_normalization_histogram(original_folder, preprocessed_folder):

    def compute_mean_brightness(folder):
        brightness_values = []

        for path in sorted(Path(folder).glob("*.jpg")):
            img = cv2.imread(str(path))
            if img is None:
                continue

            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
            v_channel = hsv[:, :, 2]

            mean_brightness = np.mean(v_channel)
            brightness_values.append(mean_brightness)

        return brightness_values

    original_vals = compute_mean_brightness(original_folder)
    preprocessed_vals = compute_mean_brightness(preprocessed_folder)

    df = pd.DataFrame({
        "Mean Brightness": original_vals + preprocessed_vals,
        "Image Type": ["Original"] * len(original_vals) +
                      ["Preprocessed"] * len(preprocessed_vals)
    })

    fig = px.histogram(
        df,
        x="Mean Brightness",
        color="Image Type",
        barmode="overlay",
        opacity=0.6,
        nbins=30,
        title="Mean Brightness Distribution: Original vs Normalized Images"
    )

    fig.show()


compare_normalization_histogram(
    r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2",
    r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images"
)

In [1]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px

def compare_normalization_histogram(original_folder, preprocessed_folder):

    def compute_image_brightness(folder):
        brightness_values = []

        for path in sorted(Path(folder).glob("*.jpg")):
            img = cv2.imread(str(path))
            if img is None:
                continue

            hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
            v_channel = hsv[:, :, 2]

            mean_brightness = np.mean(v_channel)   # one value per image
            brightness_values.append(mean_brightness)

        return brightness_values

    original_vals = compute_image_brightness(original_folder)
    preprocessed_vals = compute_image_brightness(preprocessed_folder)

    df = pd.DataFrame({
        "Brightness": original_vals + preprocessed_vals,
        "Image Type": ["Original"] * len(original_vals) +
                      ["Preprocessed"] * len(preprocessed_vals)
    })

    fig = px.histogram(
        df,
        x="Brightness",
        color="Image Type",
        barmode="overlay",
        opacity=0.6,
        nbins=40,
        title="Brightness Distribution: Original vs Normalized Images"
    )

    fig.show()


compare_normalization_histogram(
    r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2",
    r"F:\strwaberry images\back row upated with text\Cropped_Images\down_2\preprocessed_images"
)